# Winger Dataset Enrichment

## Extending the Existing FBref Dataset

The existing winger dataset provides the **canonical population** for this project, but it does not contain every variable needed to describe attacking behaviour in sufficient detail.

Several potentially important dimensions of winger play are absent or only partially represented, including:

- territorial involvement through touches in different areas of the pitch;
- attacking penalty-area occupation;
- progressive receiving;
- total receiving volume;
- crossing behaviour;
- passing-length distribution;
- total carrying involvement and carrying distance;
- shot distance and shot-quality information;
- expected chance creation;
- the mechanisms through which shot-creating actions are generated;
- aerial involvement.

A second historical dataset containing additional FBref-derived statistics was therefore introduced as an **enrichment source**.

The purpose of this dataset is **not** to replace the previously audited FBref data or redefine the winger population.

Instead:

> **The existing winger dataset remains the authoritative population, while the second dataset acts only as a feature donor.**

This distinction is important because winger eligibility, positional classification and the playing-time threshold have already been established independently.

The objective of this notebook is therefore:

> **Preserve the existing player × club × season winger population while enriching each observation with additional behavioural variables from another FBref-derived dataset.**

This notebook stops after producing and validating the enriched dataset.  
Actual playstyle feature engineering will be handled separately in `04_playstyle_feature_engineering.ipynb`.

In [ ]:
import pandas as pd

In [ ]:
wingers_df = pd.read_csv("/content/fbref_2017_2024_wingers.csv")

In [ ]:
df2 = pd.read_excel("/content/Top5_League_Players_2017to2024_dataset.xlsx")

## 1. Initial Inspection of the Donor Dataset

Before deciding which variables to merge, the donor dataset is inspected in its raw form.

At this point no attempt is made to restrict the data to wingers or even to the seasons used in the existing project. The objective is simply to understand:

- the observation structure;
- season and league encodings;
- player and club naming conventions;
- positional labels;
- the breadth of available statistics;
- and whether the dataset appears compatible with the existing FBref-derived data.

The donor dataset contains **178 columns**, substantially more than the existing winger dataset. Rather than selecting variables immediately, a sample is first inspected to understand the structure of the source.

In [ ]:
df2.sample(n=10)

,league,season,team,player,nation_,pos_,age_,born_,Playing Time_MP,Playing Time_Starts,...,Performance_Crs,Performance_Fld,Performance_Fls,Performance_Int,Performance_OG,Performance_Off,Performance_PKcon,Performance_PKwon,Performance_Recov,Performance_TklW
8726,ESP-La Liga,2425,Las Palmas,Enzo Loiodice,FRA,MF,23.0,2000.0,17,7,...,10.0,6.0,10,10.0,0.0,1.0,0.0,0.0,29.0,7.0
21843,ITA-Serie A,2324,Frosinone,Kaio Jorge,BRA,FW,21.0,2002.0,20,8,...,3.0,10.0,8,1.0,0.0,2.0,0.0,0.0,29.0,4.0
294,ENG-Premier League,1718,Manchester Utd,Chris Smalling,ENG,DF,27.0,1989.0,29,28,...,0.0,14.0,24,63.0,0.0,2.0,0.0,0.0,139.0,25.0
13162,FRA-Ligue 1,2425,Lens,Abdukodir Khusanov,UZB,DF,20.0,2004.0,13,11,...,5.0,3.0,13,26.0,0.0,0.0,1.0,0.0,50.0,11.0
17092,GER-Bundesliga,2324,Wolfsburg,Nicolas Cozza,FRA,DF,24.0,1999.0,6,3,...,4.0,0.0,1,4.0,0.0,0.0,0.0,0.0,12.0,6.0
15177,GER-Bundesliga,2021,Eint Frankfurt,Sebastian Rode,GER,MF,29.0,1990.0,27,19,...,14.0,19.0,30,47.0,0.0,0.0,0.0,0.0,101.0,25.0
10234,FRA-Ligue 1,1920,Bordeaux,Pablo,BRA,DF,28.0,1991.0,25,25,...,0.0,11.0,26,40.0,0.0,1.0,2.0,0.0,134.0,15.0
8093,ESP-La Liga,2324,Getafe,Carles Aleñá,ESP,MF,25.0,1998.0,29,10,...,33.0,18.0,22,4.0,0.0,3.0,0.0,0.0,53.0,8.0
7150,ESP-La Liga,2122,Sevilla,Anthony Martial,FRA,"FW,MF",25.0,1995.0,9,7,...,6.0,11.0,5,0.0,0.0,0.0,0.0,0.0,7.0,1.0
22635,ITA-Serie A,2425,Lazio,Oliver Nielsen,DEN,DF,21.0,2003.0,2,0,...,0.0,0.0,2,1.0,0.0,1.0,0.0,0.0,1.0,0.0


### Inspecting the Full Schema

The complete column list and data types are inspected next.

It reveals what additional information is genuinely available beyond the original dataset.

And, it helps identify possible ingestion problems before merging. In particular, statistics that should logically be numeric may sometimes appear as `object` columns because of formatting differences in the source file.

No cleaning is performed yet. At this stage the goal is discovery.

In [ ]:
print(df2.dtypes.to_string())

league                        object
season                         int64
team                          object
player                        object
nation_                       object
pos_                          object
age_                         float64
born_                        float64
Playing Time_MP                int64
Playing Time_Starts            int64
Playing Time_Min               int64
Playing Time_90s              object
Performance_Gls                int64
Performance_Ast                int64
Performance_G+A                int64
Performance_G-PK               int64
Performance_PK                 int64
Performance_PKatt              int64
Performance_CrdY               int64
Performance_CrdR               int64
Expected_xG                   object
Expected_npxG                 object
Expected_xAG                  object
Expected_npxG+xAG             object
Progression_PrgC             float64
Progression_PrgP             float64
Progression_PrgR             float64
P

## 2. Exploring Candidate Enrichment Variables

With 178 available columns, importing everything without understanding it would make the enrichment process unnecessarily opaque.

The next cells therefore inspect small random samples of conceptually useful variable families.

The Premier League is used only as a convenient subset for visual inspection as I am well versed with the players in the EPL and can gauge the data mentally to understand the fields better. The eventual donor dataset will still contain all five leagues.

### Identity and merge fields

The first group contains information that may help connect the two datasets:

- league;
- season;
- team;
- player;
- nationality;
- birth year;
- minutes played.

These fields are important because the datasets do not share a common source ID. A reliable merge will therefore need to reconstruct the established **player × club × season** observation using shared identity information.

The sample also immediately reveals some formatting differences from the canonical dataset. For example, league and season values use different conventions in the two sources. These differences will need to be standardized before matching.

In [ ]:
df2[df2["league"]=="ENG-Premier League"][[
    # merge identity / validation
    "league",
    "season",
    "team",
    "player",
    "nation_",
    "born_",
    "Playing Time_Min"
]].sample(n=10)

,league,season,team,player,nation_,born_,Playing Time_Min
3093,ENG-Premier League,2223,Southampton,Ibrahima Diallo,FRA,1999.0,819
188,ENG-Premier League,1718,Huddersfield,Christopher Schindler,GER,1990.0,3265
2694,ENG-Premier League,2223,Bournemouth,David Brooks,WAL,1997.0,173
2207,ENG-Premier League,2122,Burnley,Ben Mee,ENG,1989.0,1839
564,ENG-Premier League,1819,Bournemouth,Dan Gosling,ENG,1990.0,1739
769,ENG-Premier League,1819,Huddersfield,Isaac Mbenza,BEL,1996.0,986
3028,ENG-Premier League,2223,Newcastle Utd,Callum Wilson,ENG,1992.0,1877
3204,ENG-Premier League,2223,Wolves,Sasa Kalajdzic,AUT,1997.0,45
2569,ENG-Premier League,2122,Watford,Imran Louza,MAR,1999.0,1416
3381,ENG-Premier League,2324,Chelsea,Alex Matos,ENG,2004.0,1


### Touches, Territory, Receiving and Passing Choice

One of the main reasons for considering the donor dataset is that it provides information that the original dataset cannot reconstruct.

#### Touch geography

The donor contains:

- total touches;
- live-ball touches;
- defensive-third touches;
- middle-third touches;
- attacking-third touches;
- attacking penalty-area touches.

These variables are valuable because they describe **where a player's involvement occurs**, not simply how many attacking actions he completes.

They may later allow distinctions such as:

- high-volume vs low-volume involvement;
- wide build-up involvement vs advanced involvement;
- frequent penalty-area occupation vs predominantly outside-the-box play.

They also provide possible denominators for later **touch-adjusted behavioural metrics**.

#### Receiving

Total receptions and progressive receptions introduce an off-ball dimension that is largely missing from the original dataset.

A winger who repeatedly receives progressive passes may play a very different role from one who primarily progresses the ball through his own carrying or passing.

#### Passing choice

The donor also contains actual short-, medium- and long-pass **attempt counts**, rather than only completion percentages.

That is important for playstyle analysis because:

> **Pass attempts describe choice; completion percentage primarily describes execution.**

Crosses, through balls and switches of play are also available and may provide useful descriptors of passing style.

In [ ]:
df2[df2["league"]=="ENG-Premier League"][[
    "league",
    "season",
    "team",
    "player",
    # touches / territory
    "Touches_Touches",
    "Touches_Live",
    "Touches_Def 3rd",
    "Touches_Mid 3rd",
    "Touches_Att 3rd",
    "Touches_Att Pen"
]].sample(n=10)

,league,season,team,player,Touches_Touches,Touches_Live,Touches_Def 3rd,Touches_Mid 3rd,Touches_Att 3rd,Touches_Att Pen
1488,ENG-Premier League,1920,Watford,Christian Kabasele,954.0,954.0,546.0,390.0,23.0,18.0
1589,ENG-Premier League,2021,Aston Villa,Anwar El Ghazi,841.0,837.0,71.0,279.0,498.0,72.0
3753,ENG-Premier League,2324,West Ham,Pablo Fornals,277.0,277.0,53.0,128.0,98.0,20.0
793,ENG-Premier League,1819,Leicester City,Hamza Choudhury,328.0,328.0,92.0,185.0,56.0,2.0
3380,ENG-Premier League,2324,Burnley,Zeki Amdouni,789.0,788.0,94.0,379.0,337.0,68.0
2107,ENG-Premier League,2122,Arsenal,Martin Ødegaard,1867.0,1867.0,197.0,886.0,796.0,59.0
2896,ENG-Premier League,2223,Leeds United,Georginio Rutter,124.0,124.0,13.0,47.0,65.0,20.0
428,ENG-Premier League,1718,Tottenham,Dele Alli,1726.0,1726.0,179.0,702.0,874.0,147.0
4275,ENG-Premier League,2425,Tottenham,Brennan Johnson,853.0,853.0,98.0,292.0,475.0,125.0
617,ENG-Premier League,1819,Burnley,Joe Hart,685.0,685.0,648.0,38.0,0.0,0.0


In [ ]:
df2[df2["league"]=="ENG-Premier League"][[
    "league",
    "season",
    "team",
    "player",
    # receiving
    "Receiving_Rec",
    "Receiving_PrgR"
]].sample(n=10)

,league,season,team,player,Receiving_Rec,Receiving_PrgR
2092,ENG-Premier League,2122,Arsenal,Ainsley Maitland-Niles,110.0,8.0
1050,ENG-Premier League,1920,Arsenal,Henrikh Mkhitaryan,75.0,19.0
1176,ENG-Premier League,1920,Chelsea,Faustino Anjorin,8.0,1.0
2547,ENG-Premier League,2122,Tottenham,Matt Doherty,420.0,80.0
4223,ENG-Premier League,2425,Nott'ham Forest,Morgan Gibbs-White,1226.0,136.0
3263,ENG-Premier League,2324,Bournemouth,Alex Scott,361.0,35.0
525,ENG-Premier League,1718,West Ham,Pedro Obiang,584.0,10.0
569,ENG-Premier League,1819,Bournemouth,Jack Simpson,168.0,0.0
2892,ENG-Premier League,2223,Leeds United,Crysencio Summerville,438.0,92.0
540,ENG-Premier League,1819,Arsenal,Henrikh Mkhitaryan,782.0,144.0


In [ ]:
df2[df2["league"]=="ENG-Premier League"][[
    "league",
    "season",
    "team",
    "player",
    # passing profile
    "Total_Att",
    "Short_Att",
    "Medium_Att",
    "Long_Att",
    "Pass Types_Crs",
    "Pass Types_TB",
    "Pass Types_Sw",
    "CrsPA_"
]].sample(n=10)

,league,season,team,player,Total_Att,Short_Att,Medium_Att,Long_Att,Pass Types_Crs,Pass Types_TB,Pass Types_Sw,CrsPA_
2925,ENG-Premier League,2223,Leicester City,Harvey Barnes,786.0,462.0,210.0,37.0,44.0,5.0,1.0,5.0
457,ENG-Premier League,1718,Watford,Dodi Lukebakio,8.0,2.0,4.0,1.0,2.0,0.0,0.0,1.0
3558,ENG-Premier League,2324,Manchester City,Julián Álvarez,1258.0,679.0,328.0,145.0,137.0,9.0,1.0,4.0
1281,ENG-Premier League,1920,Liverpool,Jordan Henderson,1933.0,759.0,769.0,289.0,58.0,7.0,21.0,7.0
1683,ENG-Premier League,2021,Chelsea,Olivier Giroud,223.0,138.0,25.0,5.0,2.0,0.0,0.0,0.0
1889,ENG-Premier League,2021,Manchester Utd,Hannibal Mejbri,3.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0
347,ENG-Premier League,1718,Southampton,Guido Carrillo,124.0,74.0,27.0,3.0,0.0,1.0,0.0,0.0
2023,ENG-Premier League,2021,West Brom,Jake Livermore,388.0,170.0,141.0,48.0,5.0,1.0,3.0,1.0
1337,ENG-Premier League,1920,Manchester Utd,Nemanja Matić,1273.0,583.0,482.0,130.0,3.0,2.0,8.0,2.0
3582,ENG-Premier League,2324,Manchester Utd,Daniel Gore,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Carrying, Shooting and Chance-Creation Detail

The donor dataset also expands several areas already represented in the original data.

#### Carrying

The original dataset contains progressive carries and carries into important attacking zones.

The donor adds:

- total carries;
- total carrying distance;
- progressive carrying distance;
- miscontrols;
- dispossessions.

These variables may later help distinguish **overall carrying involvement** from specifically progressive or penetrative carrying.

#### Shooting profile

Average shot distance and non-penalty expected goals per shot provide information about **shot selection**, rather than simply goals or conversion.

They are not automatically assumed to be playstyle features, but they are worth retaining for later evaluation.

#### Expected chance creation

The donor contains `xAG` and `xA`, providing additional ways to evaluate chance creation without relying solely on actual assists.

Again, these may eventually be more useful for interpreting quality or effectiveness than for defining style itself. The current goal is only to preserve the information.

In [ ]:
df2[df2["league"]=="ENG-Premier League"][[
    "league",
    "season",
    "team",
    "player",
    # carrying
    "Carries_Carries",
    "Carries_PrgDist",
    "Carries_TotDist",
    "Carries_Mis"
]].sample(n=10)

,league,season,team,player,Carries_Carries,Carries_PrgDist,Carries_TotDist,Carries_Mis
673,ENG-Premier League,1819,Chelsea,Ruben Loftus-Cheek,502.0,1761.0,3122.0,25.0
1913,ENG-Premier League,2021,Newcastle Utd,Isaac Hayden,465.0,578.0,1597.0,20.0
358,ENG-Premier League,1718,Southampton,Nathan Redmond,781.0,2670.0,5230.0,38.0
1546,ENG-Premier League,1920,Wolves,Leander Dendoncker,767.0,1941.0,3409.0,31.0
4144,ENG-Premier League,2425,Manchester City,Nico O’Reilly,249.0,575.0,1171.0,14.0
442,ENG-Premier League,1718,Tottenham,Paulo Gazzaniga,9.0,17.0,25.0,0.0
1102,ENG-Premier League,1920,Bournemouth,David Brooks,137.0,240.0,700.0,7.0
1001,ENG-Premier League,1819,West Ham,Issa Diop,679.0,2369.0,3927.0,13.0
3339,ENG-Premier League,2324,Brighton,Kaoru Mitoma,616.0,3147.0,4879.0,38.0
3827,ENG-Premier League,2425,Aston Villa,Leon Bailey,348.0,1371.0,2582.0,35.0


In [ ]:
df2[df2["league"]=="ENG-Premier League"][[
    "league",
    "season",
    "team",
    "player",
    # shooting profile
    "Standard_Dist",
    "Standard_FK",
    "Expected_npxG/Sh",

    # chance creation quality
    "Expected_xAG",
]].sample(n=10)

,league,season,team,player,Standard_Dist,Standard_FK,Expected_npxG/Sh,Expected_xAG
3646,ENG-Premier League,2324,Nott'ham Forest,Felipe,"16,6",0.0,"0,04","0,1"
2369,ENG-Premier League,2122,Liverpool,Andrew Robertson,"16,9",0.0,"0,08","5,6"
983,ENG-Premier League,1819,Watford,Roberto Pereyra,"17,5",6.0,"0,09","3,2"
2352,ENG-Premier League,2122,Leicester City,Kasper Schmeichel,NaN,0.0,NaN,0
2118,ENG-Premier League,2122,Aston Villa,Anwar El Ghazi,"22,3",0.0,"0,04","0,4"
3776,ENG-Premier League,2324,Wolves,Nathan Fraser,"16,6",0.0,"0,11","0,1"
3107,ENG-Premier League,2223,Southampton,Moussa Djenepo,"13,8",0.0,"0,06","0,5"
1741,ENG-Premier League,2021,Everton,Theo Walcott,NaN,0.0,NaN,0
169,ENG-Premier League,1718,Everton,Leighton Baines,"24,7",2.0,"0,1","1,9"
1851,ENG-Premier League,2021,Manchester City,Benjamin Mendy,"17,2",0.0,"0,06","0,2"


### Mechanism-Level and Secondary Behavioural Variables

Several additional groups are explored because they may help interpret winger roles later.

#### Shot-Creating Action types

Rather than using only total Shot-Creating Actions, the donor separates the actions preceding shots into mechanisms such as:

- live-ball passes;
- dead-ball passes;
- take-ons;
- shots;
- fouls drawn.

These variables are somewhat downstream because they depend on a shot eventually occurring, so they are **not automatically treated as primary playstyle features**. However, they may provide a useful interpretation layer later.

#### Aerial involvement

The original dataset contains aerial-duel win percentage, which mainly reflects success.

The donor contains aerial duels won and lost, allowing total aerial involvement to be reconstructed. That could distinguish wingers who are frequently involved in aerial situations from those who rarely contest them.

#### High defensive activity

Attacking-third tackles are retained as a possible indicator of defensive intervention high up the pitch.

This should **not** be interpreted as a complete pressing metric: pressing can occur without a tackle being recorded.

#### Take-on diagnostics

The donor contains both raw take-on counts and take-on percentages.

These are particularly useful because the original data audit identified a possible historical definition change around take-on outcomes. Retaining the underlying counts gives us another way to investigate those metrics before trusting success percentages in modelling.

In [ ]:
df2[df2["league"]=="ENG-Premier League"][[
    "league",
    "season",
    "team",
    "player",
    # SCA mechanism — interpretation
    "SCA Types_PassLive",
    "SCA Types_PassDead",
    "SCA Types_TO",
    "SCA Types_Sh",
    "SCA Types_Fld"
]].sample(n=10)

,league,season,team,player,SCA Types_PassLive,SCA Types_PassDead,SCA Types_TO,SCA Types_Sh,SCA Types_Fld
2485,ENG-Premier League,2122,Norwich City,Christoph Zimmermann,0.0,0.0,1.0,0.0,0.0
1794,ENG-Premier League,2021,Leeds United,Tyler Roberts,29.0,0.0,3.0,1.0,6.0
1609,ENG-Premier League,2021,Aston Villa,Trézéguet,24.0,0.0,2.0,3.0,5.0
1112,ENG-Premier League,1920,Bournemouth,Lewis Cook,31.0,2.0,0.0,5.0,2.0
4211,ENG-Premier League,2425,Nott'ham Forest,Anthony Elanga,56.0,23.0,5.0,7.0,11.0
929,ENG-Premier League,1819,Southampton,Ryan Bertrand,36.0,11.0,0.0,1.0,4.0
3052,ENG-Premier League,2223,Nott'ham Forest,André Ayew,2.0,1.0,0.0,0.0,0.0
2425,ENG-Premier League,2122,Manchester Utd,Bruno Fernandes,115.0,19.0,5.0,10.0,5.0
3535,ENG-Premier League,2324,Luton Town,Joseph Johnson,0.0,0.0,0.0,0.0,0.0
3141,ENG-Premier League,2223,Tottenham,Pape Matar Sarr,4.0,0.0,0.0,0.0,0.0


In [ ]:
df2[df2["league"]=="ENG-Premier League"][[
    "league",
    "season",
    "team",
    "player",

    # aerial involvement
    "Aerial Duels_Won",
    "Aerial Duels_Lost"
]].sample(n=10)

,league,season,team,player,Aerial Duels_Won,Aerial Duels_Lost
736,ENG-Premier League,1819,Fulham,Floyd Ayité,6.0,7.0
3693,ENG-Premier League,2324,Sheffield Utd,Oliver Norwood,14.0,16.0
59,ENG-Premier League,1718,Brighton,Gaëtan Bong,29.0,28.0
1932,ENG-Premier League,2021,Sheffield Utd,Ben Osborn,12.0,17.0
3655,ENG-Premier League,2324,Nott'ham Forest,Moussa Niakhate,15.0,5.0
1439,ENG-Premier League,1920,Southampton,Kevin Danso,7.0,12.0
704,ENG-Premier League,1819,Everton,Ademola Lookman,6.0,5.0
2861,ENG-Premier League,2223,Fulham,Andreas Pereira,9.0,28.0
4135,ENG-Premier League,2425,Manchester City,Jeremy Doku,0.0,12.0
3954,ENG-Premier League,2425,Chelsea,Trevoh Chalobah,23.0,12.0


In [ ]:
df2[df2["league"]=="ENG-Premier League"][[
    "league",
    "season",
    "team",
    "player",

    # high defensive activity
    "Tackles_Att 3rd"
]].sample(n=10)

,league,season,team,player,Tackles_Att 3rd
4218,ENG-Premier League,2425,Nott'ham Forest,Ibrahim Sangaré,3.0
724,ENG-Premier League,1819,Everton,Theo Walcott,7.0
1272,ENG-Premier League,1920,Liverpool,Andrew Robertson,12.0
875,ENG-Premier League,1819,Manchester Utd,Matteo Darmian,0.0
537,ENG-Premier League,1819,Arsenal,Denis Suárez,0.0
2782,ENG-Premier League,2223,Chelsea,Denis Zakaria,2.0
2672,ENG-Premier League,2223,Aston Villa,Emiliano Martínez,0.0
3412,ENG-Premier League,2324,Chelsea,Đorđe Petrović,0.0
3054,ENG-Premier League,2223,Nott'ham Forest,Cafú,0.0
4117,ENG-Premier League,2425,Liverpool,Joe Gomez,1.0


In [ ]:
df2[df2["league"]=="ENG-Premier League"][[
    "league",
    "season",
    "team",
    "player",

    # temporary take-on diagnostics
    "Take-Ons_Att",
    "Take-Ons_Succ",
    "Take-Ons_Tkld",
    "Take-Ons_Succ%",
    "Take-Ons_Tkld%"
]].sample(n=10)

,league,season,team,player,Take-Ons_Att,Take-Ons_Succ,Take-Ons_Tkld,Take-Ons_Succ%,Take-Ons_Tkld%
311,ENG-Premier League,1718,Manchester Utd,Scott McTominay,8.0,5.0,3.0,"62,5","37,5"
1852,ENG-Premier League,2021,Manchester City,Bernardo Silva,65.0,41.0,24.0,"63,1","36,9"
1447,ENG-Premier League,1920,Southampton,Pierre Højbjerg,52.0,40.0,12.0,"76,9","23,1"
3389,ENG-Premier League,2324,Chelsea,Christopher Nkunku,8.0,2.0,5.0,25,"62,5"
3807,ENG-Premier League,2425,Arsenal,Riccardo Calafiori,13.0,9.0,3.0,"69,2","23,1"
2961,ENG-Premier League,2223,Liverpool,Joe Gomez,5.0,4.0,1.0,80,20
2398,ENG-Premier League,2122,Manchester City,Cole Palmer,2.0,1.0,1.0,50,50
3557,ENG-Premier League,2324,Manchester City,Joško Gvardiol,45.0,26.0,14.0,"57,8","31,1"
1165,ENG-Premier League,1920,Burnley,Nick Pope,0.0,0.0,0.0,NaN,NaN
434,ENG-Premier League,1718,Tottenham,Hugo Lloris,1.0,1.0,0.0,100,0


### Overlapping Variables as an Audit Tool

Several donor fields duplicate statistics already present in the canonical winger dataset.

These include:

- goals;
- assists;
- progressive carries;
- progressive passes;
- key passes;
- passes into the final third;
- passes into the penalty area;
- carries into the final third;
- carries into the penalty area;
- total shots.

Normally there would be no reason to retain duplicate columns.

Here, however, they are deliberately kept **temporarily**.

Because both datasets contain independently scraped versions of the same FBref statistics, these overlapping variables can act as a powerful validation mechanism after the merge.

If the correct player-club-season records are matched, most duplicated statistics should agree very closely.

Large disagreements can therefore expose either:

- an incorrect merge;
- historical source revisions;
- or an error in one of the datasets.

In [ ]:
df2[df2["league"]=="ENG-Premier League"][[
    "league",
    "season",
    "team",
    "player",

    # duplicate audit fields
    "Performance_Gls",
    "Performance_Ast",
    "Progression_PrgC",
    "Progression_PrgP",
    "KP_",
    "1/3_",
    "PPA_",
    "Carries_1/3",
    "Carries_CPA",
    "Standard_Sh"
]].sample(n=10)

,league,season,team,player,Performance_Gls,Performance_Ast,Progression_PrgC,Progression_PrgP,KP_,1/3_,PPA_,Carries_1/3,Carries_CPA,Standard_Sh
3745,ENG-Premier League,2324,West Ham,Kalvin Phillips,0,0,3.0,19.0,0.0,24.0,0.0,1.0,0.0,3.0
2997,ENG-Premier League,2223,Manchester City,İlkay Gündoğan,8,4,53.0,170.0,37.0,143.0,35.0,44.0,7.0,53.0
3418,ENG-Premier League,2324,Crystal Palace,Dean Henderson,0,0,0.0,0.0,0.0,9.0,1.0,0.0,0.0,0.0
4355,ENG-Premier League,2425,Wolves,Santiago Bueno,0,0,1.0,28.0,0.0,24.0,2.0,2.0,0.0,6.0
1705,ENG-Premier League,2021,Crystal Palace,Luka Milivojević,1,1,12.0,120.0,21.0,117.0,10.0,13.0,2.0,17.0
3903,ENG-Premier League,2425,Brighton,Eiran Cashin,0,0,0.0,6.0,0.0,5.0,0.0,0.0,0.0,0.0
3052,ENG-Premier League,2223,Nott'ham Forest,André Ayew,0,0,3.0,11.0,1.0,3.0,1.0,3.0,2.0,3.0
1640,ENG-Premier League,2021,Burnley,Ashley Westwood,3,3,21.0,231.0,53.0,204.0,83.0,25.0,2.0,20.0
4211,ENG-Premier League,2425,Nott'ham Forest,Anthony Elanga,6,11,102.0,63.0,50.0,32.0,26.0,45.0,48.0,44.0
640,ENG-Premier League,1819,Cardiff City,Josh Murphy,3,2,81.0,23.0,16.0,16.0,15.0,44.0,32.0,46.0


## 3. Constructing the Working Donor Dataset

The exploratory inspection suggests that the new source adds several genuinely useful dimensions without requiring the existing winger population to be rebuilt.

The enrichment strategy is therefore:

1. retain the previously established winger dataset as canonical;
2. reduce the donor source to a relevant comparison population;
3. retain useful behavioural variables plus temporary audit fields;
4. establish an exact and defensible join;
5. validate overlapping statistics after the merge.

The next stage prepares the donor dataset for that purpose.

In [ ]:
# Working on a copy
donor_df = df2.copy()

print("Starting shape:", donor_df.shape)

Starting shape: (22929, 178)


### Aligning the Donor Population

The raw donor contains **22,929 observations across 178 columns**, including:

- seasons outside the current project;
- defenders and goalkeepers;
- players with very limited minutes.

For the initial matching exercise, the donor population is reduced to something broadly comparable with the candidate population used when constructing the winger dataset.

This is only a **matching convenience**.

The donor dataset does not determine who qualifies as a winger.

In [ ]:
# Example:
# "2017-2018" -> 1718
# "2018-2019" -> 1819

def season_to_donor_code(season):
    start, end = str(season).split("-")
    return int(start[-2:] + end[-2:])


valid_donor_seasons = (
    wingers_df["season"]
    .dropna()
    .unique()
)

valid_donor_seasons = {
    season_to_donor_code(season)
    for season in valid_donor_seasons
}

print("Donor season codes to keep:")
print(sorted(valid_donor_seasons))

Donor season codes to keep:
[1718, 1819, 1920, 2021, 2122, 2223, 2324]


In [ ]:
donor_df = donor_df[
    donor_df["season"].isin(valid_donor_seasons)
].copy()

print("After season filter:", donor_df.shape)

After season filter: (20075, 178)


In [ ]:
donor_df["pos_"].isin(["MF", "FW"])

,pos_
0,True
1,False
2,False
3,False
4,True
...,...
22290,True
22291,True
22292,False
22293,True


In [ ]:
def is_mf_or_fw(pos):
    if pd.isna(pos):
        return False

    positions = {
        p.strip()
        for p in str(pos).split(",")
    }

    return bool(positions & {"MF", "FW"})


donor_df = donor_df[
    donor_df["pos_"].apply(is_mf_or_fw)
].copy()

print("After MF/FW filter:", donor_df.shape)

print("\nRemaining position values:")
print(donor_df["pos_"].value_counts(dropna=False))

After MF/FW filter: (12751, 178)

Remaining position values:
pos_
MF       4668
FW       2653
FW,MF    2046
MF,FW    1756
DF,MF     685
MF,DF     465
DF,FW     321
FW,DF     157
Name: count, dtype: int64


In [ ]:
MIN_MINUTES = 1125

donor_df = donor_df[
    donor_df["Playing Time_Min"] >= MIN_MINUTES
].copy()

print("After 1,125-minute filter:", donor_df.shape)

After 1,125-minute filter: (5689, 178)


### Population Filtering Results

The donor population was reduced in three stages:

- relevant project seasons only;
- observations containing `MF` or `FW` in the broad position field;
- at least **1,125 minutes** played.

The resulting donor population contains:

> **5,689 observations**

Hybrid labels such as `DF,MF`, `MF,DF`, `DF,FW` and `FW,DF` were deliberately retained when they contained `MF` or `FW`.

This broad filter should not be confused with the detailed winger classification performed previously using FIFA and manual historical positional verification.

Its only purpose here is to reduce irrelevant donor observations while preserving plausible matches.

In [ ]:
donor_columns = [

    # =====================================================
    # IDENTITY / MERGE / AUDIT
    # =====================================================
    "league",
    "season",
    "team",
    "player",
    "nation_",
    "pos_",
    "age_",
    "born_",
    "Playing Time_MP",
    "Playing Time_Starts",
    "Playing Time_Min",
    "Playing Time_90s",

    # =====================================================
    # TOUCHES / TERRITORY
    # =====================================================
    "Touches_Touches",
    "Touches_Live",
    "Touches_Def 3rd",
    "Touches_Mid 3rd",
    "Touches_Att 3rd",
    "Touches_Att Pen",

    # =====================================================
    # RECEIVING
    # =====================================================
    "Receiving_Rec",
    "Receiving_PrgR",
    "Progression_PrgR",

    # =====================================================
    # PASSING PROFILE
    # =====================================================
    "Total_Att",
    "Short_Att",
    "Medium_Att",
    "Long_Att",
    "Pass Types_Crs",
    "Pass Types_TB",
    "Pass Types_Sw",
    "CrsPA_",

    # =====================================================
    # CARRYING
    # =====================================================
    "Carries_Carries",
    "Carries_PrgDist",
    "Carries_TotDist",
    "Carries_Mis",
    "Carries_Dis",

    # =====================================================
    # SHOOTING PROFILE
    # =====================================================
    "Standard_Dist",
    "Standard_FK",
    "Expected_npxG/Sh",

    # =====================================================
    # CHANCE CREATION / EXPECTED CREATION
    # =====================================================
    "Expected_xAG",
    "Expected_xA",

    # =====================================================
    # SCA MECHANISM
    # =====================================================
    "SCA Types_PassLive",
    "SCA Types_PassDead",
    "SCA Types_TO",
    "SCA Types_Sh",
    "SCA Types_Fld",
    "SCA Types_Def",
    "SCA_SCA",

    # =====================================================
    # AERIAL INVOLVEMENT
    # =====================================================
    "Aerial Duels_Won",
    "Aerial Duels_Lost",

    # =====================================================
    # HIGH-PITCH DEFENSIVE ACTIVITY
    # =====================================================
    "Tackles_Att 3rd",

    # =====================================================
    # TAKE-ON RAW DATA / HISTORICAL DIAGNOSTICS
    # =====================================================
    "Take-Ons_Att",
    "Take-Ons_Succ",
    "Take-Ons_Succ%",
    "Take-Ons_Tkld",
    "Take-Ons_Tkld%",

    # =====================================================
    # DUPLICATE FIELDS FOR MERGE VALIDATION
    # =====================================================
    "Performance_Gls",
    "Performance_Ast",
    "Progression_PrgC",
    "Progression_PrgP",
    "KP_",
    "1/3_",
    "PPA_",
    "Carries_1/3",
    "Carries_CPA",
    "Standard_Sh"
]

In [ ]:
donor_df = donor_df[donor_columns].copy()

print("Final donor shape:", donor_df.shape)
print(f"Columns retained: {donor_df.shape[1]}")

Final donor shape: (5689, 64)
Columns retained: 64


In [ ]:
donor_df = donor_df.reset_index(drop=True)

In [ ]:
display(donor_df.head())

print(donor_df.shape)
print(donor_df["season"].value_counts().sort_index())

print("\nPositions:")
print(donor_df["pos_"].value_counts())

,league,season,team,player,nation_,pos_,age_,born_,Playing Time_MP,Playing Time_Starts,...,Performance_Gls,Performance_Ast,Progression_PrgC,Progression_PrgP,KP_,1/3_,PPA_,Carries_1/3,Carries_CPA,Standard_Sh
0,ENG-Premier League,1718,Arsenal,Aaron Ramsey,WAL,MF,26.0,1990.0,24,21,...,7,8,61.0,134.0,28.0,102.0,36.0,54.0,18.0,56.0
1,ENG-Premier League,1718,Arsenal,Alex Iwobi,NGA,"MF,FW",21.0,1996.0,26,22,...,3,5,93.0,138.0,36.0,93.0,31.0,80.0,25.0,45.0
2,ENG-Premier League,1718,Arsenal,Alexandre Lacazette,FRA,FW,26.0,1991.0,32,26,...,14,4,39.0,59.0,36.0,18.0,31.0,12.0,27.0,66.0
3,ENG-Premier League,1718,Arsenal,Alexis Sánchez,CHI,"MF,FW",28.0,1988.0,19,17,...,7,3,50.0,149.0,50.0,56.0,74.0,53.0,25.0,68.0
4,ENG-Premier League,1718,Arsenal,Danny Welbeck,ENG,"FW,MF",26.0,1990.0,28,12,...,5,2,30.0,23.0,13.0,7.0,9.0,20.0,21.0,32.0


(5689, 64)
season
1718    812
1819    829
1920    783
2021    846
2122    818
2223    800
2324    801
Name: count, dtype: int64

Positions:
pos_
MF       2156
FW       1172
FW,MF     850
MF,FW     773
DF,MF     320
MF,DF     219
DF,FW     154
FW,DF      45
Name: count, dtype: int64


### Working Donor Feature Set

After the exploratory review, **64 columns** are retained.

The retained fields include three different types of information:

1. **new behavioural information** that may be useful later;
2. **identity/context fields** required for matching;
3. **duplicate audit fields** retained temporarily to validate the merge.

This is intentionally broader than the final modelling feature set.

No claim is being made that all 64 columns belong in a playstyle model. Feature selection is deferred to the next notebook.

## 4. Verifying the Donor Observation Grain

Before attempting any merge, the donor dataset must be checked against the observation grain established earlier in the project:

> **player × club × season**

This is particularly important for transferred players.

The project deliberately preserves separate club spells because tactical context may change when the same player moves between teams during a season.

The donor is therefore checked for:

- synthetic multi-club aggregate rows;
- duplicate player × team × season observations;
- legitimate players appearing for multiple clubs during the same season.

In [ ]:
teams = sorted(donor_df["team"].dropna().unique())

[x for x in teams if "club" in x.lower()]

['Athletic Club']

In [ ]:
donor_duplicates = (
    donor_df
    .groupby(
        ["player", "team", "season"],
        dropna=False
    )
    .size()
    .reset_index(name="rows")
    .query("rows > 1")
    .sort_values("rows", ascending=False)
)

print("Duplicate player × team × season groups:")
print(len(donor_duplicates))

display(donor_duplicates.head(30))

Duplicate player × team × season groups:
0


,player,team,season,rows


In [ ]:
multi_team_players = (
    donor_df
    .groupby(["player", "season"])["team"]
    .nunique()
    .reset_index(name="number_of_teams")
    .query("number_of_teams > 1")
    .sort_values("number_of_teams", ascending=False)
)

display(multi_team_players.head(30))

,player,season,number_of_teams
808,Chris Wood,2122,2
943,Dango Ouattara,2223,2
2096,Ivan Radovanović,1819,2
2236,Jasmin Kurtić,1920,2
2593,João Pedro,2122,2
2858,Krzysztof Piątek,1819,2
3229,Manu García,1819,2
3579,Max Kruse,2122,2
3724,Milan Đurić,2324,2
3966,Nicolás González,2223,2


### Grain Audit Result

The donor dataset contains:

> **0 duplicate player × team × season groups**

A number of players appear for two clubs in the same season, but these are legitimate transfer cases.

No synthetic multi-club aggregate records were found.

The donor structure is therefore compatible with the existing methodology: individual club spells can remain separate rather than being aggregated.

## 5. Standardising the Merge Fields

The datasets describe the same competitions and many of the same players, but several shared fields use different formatting conventions.

Examples identified during exploration include:

- donor season `1819` vs canonical season `2018-2019`;
- donor league `ENG-Premier League` vs canonical `Premier League`;
- birth years stored as different numeric types;
- differences in punctuation, accents and whitespace within names.

Rather than altering the original source columns, temporary normalized fields are created for matching.

This preserves the raw source information while giving the merge a consistent representation.

In [ ]:
import re
import unicodedata

# Work from copies
wingers_merge = wingers_df.copy()
donor_merge = donor_df.copy()

In [ ]:
def donor_season_to_standard(season):
    season = str(int(season)).zfill(4)

    start_yy = int(season[:2])
    end_yy = int(season[2:])

    start_year = 2000 + start_yy
    end_year = 2000 + end_yy

    return f"{start_year}-{end_year}"


donor_merge["season_norm"] = (
    donor_merge["season"]
    .apply(donor_season_to_standard)
)

wingers_merge["season_norm"] = (
    wingers_merge["season"]
    .astype(str)
    .str.strip()
)

In [ ]:
print(sorted(wingers_merge["season_norm"].unique()))
print(sorted(donor_merge["season_norm"].unique()))

['2017-2018', '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024']
['2017-2018', '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024']


In [ ]:
donor_merge["league_norm"] = (
    donor_merge["league"]
    .astype(str)
    .str.replace(r"^[A-Z]{3}-", "", regex=True)
    .str.strip()
)

wingers_merge["league_norm"] = (
    wingers_merge["comp"]
    .astype(str)
    .str.strip()
)

In [ ]:
print(sorted(wingers_merge["league_norm"].unique()))
print(sorted(donor_merge["league_norm"].unique()))

['Bundesliga', 'La Liga', 'Ligue 1', 'Premier League', 'Serie A']
['Bundesliga', 'La Liga', 'Ligue 1', 'Premier League', 'Serie A']


In [ ]:
donor_merge["born_norm"] = pd.to_numeric(
    donor_merge["born_"],
    errors="coerce"
).astype("Int64")

wingers_merge["born_norm"] = pd.to_numeric(
    wingers_merge["born"],
    errors="coerce"
).astype("Int64")

In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    # Remove accents:
    # João -> joao
    # Sánchez -> sanchez
    value = unicodedata.normalize("NFKD", value)
    value = "".join(
        char for char in value
        if not unicodedata.combining(char)
    )

    # Standardize punctuation / separators
    value = re.sub(r"[^a-z0-9\s]", " ", value)

    # Collapse repeated whitespace
    value = re.sub(r"\s+", " ", value).strip()

    return value


wingers_merge["player_norm"] = (
    wingers_merge["player"]
    .apply(normalize_text)
)

donor_merge["player_norm"] = (
    donor_merge["player"]
    .apply(normalize_text)
)

In [ ]:
wingers_merge["club_norm_merge"] = (
    wingers_merge["squad"]
    .apply(normalize_text)
)

donor_merge["club_norm_merge"] = (
    donor_merge["team"]
    .apply(normalize_text)
)

## 6. Progressive Exact Matching

The merge is tested progressively rather than introducing fuzzy matching immediately.

The first matching key is:

> **player + birth year + season**

This tests whether the same player-season identity exists in both datasets.

The second key adds club:

> **player + birth year + season + club**

This corresponds to the actual observational grain required for enrichment.

This approach is diagnostically useful. If player-season matching succeeds but adding club causes failures, the problem is likely club naming rather than player identity.

Exact matching is preferred wherever possible because it is easier to audit and less likely to introduce false-positive identity matches.

In [ ]:
identity_keys = [
    "player_norm",
    "born_norm",
    "season_norm"
]

donor_identity = (
    donor_merge[identity_keys]
    .drop_duplicates()
)

winger_identity = (
    wingers_merge[identity_keys]
    .drop_duplicates()
)

identity_check = winger_identity.merge(
    donor_identity,
    on=identity_keys,
    how="left",
    indicator=True
)

print(
    identity_check["_merge"]
    .value_counts()
)

print(
    "\nMatch rate:",
    round(
        (identity_check["_merge"] == "both").mean() * 100,
        2
    ),
    "%"
)

_merge
both          1392
left_only        3
right_only       0
Name: count, dtype: int64

Match rate: 99.78 %


In [ ]:
full_keys = [
    "player_norm",
    "born_norm",
    "season_norm",
    "club_norm_merge"
]

donor_full = (
    donor_merge[full_keys]
    .drop_duplicates()
)

winger_full = (
    wingers_merge[full_keys]
    .drop_duplicates()
)

full_match_check = winger_full.merge(
    donor_full,
    on=full_keys,
    how="left",
    indicator=True
)

print(
    full_match_check["_merge"]
    .value_counts()
)

print(
    "\nExact player-club-season match rate:",
    round(
        (full_match_check["_merge"] == "both").mean() * 100,
        2
    ),
    "%"
)

_merge
both          1393
left_only        3
right_only       0
Name: count, dtype: int64

Exact player-club-season match rate: 99.79 %


In [ ]:
unmatched_full = (
    full_match_check[
        full_match_check["_merge"] == "left_only"
    ]
    .drop(columns="_merge")
)

print(
    "Unmatched winger observations:",
    len(unmatched_full)
)

display(unmatched_full.head(50))

Unmatched winger observations: 3


,player_norm,born_norm,season_norm,club_norm_merge
1162,bobby reid,1993,2022-2023,fulham
1243,mohammed daramy,2002,2023-2024,reims
1350,bobby reid,1993,2023-2024,fulham


### Initial Matching Result

The first player-season identity audit produced:

- **1,392 matched unique player-season identities**
- **3 unmatched identities**

The full player-club-season key produced:

- **1,393 matched observations**
- **3 unmatched observations**

The difference in total counts between the two checks is expected: removing club from the key collapses a player who appears for multiple clubs in the same season.

The important result is that only **three player-club-season observations** fail exact matching.

This is a sufficiently small number that building a generalized fuzzy-matching system would be unnecessary. The exceptions can be investigated directly.

Before resolving them, the selected donor feature columns are also checked for datatype problems.

In [ ]:
selected_numeric_cols = [
    "Touches_Touches",
    "Touches_Live",
    "Touches_Def 3rd",
    "Touches_Mid 3rd",
    "Touches_Att 3rd",
    "Touches_Att Pen",
    "Receiving_Rec",
    "Receiving_PrgR",
    "Progression_PrgR",
    "Total_Att",
    "Short_Att",
    "Medium_Att",
    "Long_Att",
    "Pass Types_Crs",
    "Pass Types_TB",
    "Pass Types_Sw",
    "CrsPA_",
    "Carries_Carries",
    "Carries_PrgDist",
    "Carries_TotDist",
    "Carries_Mis",
    "Carries_Dis",
    "Standard_Dist",
    "Standard_FK",
    "Expected_npxG/Sh",
    "Expected_xAG",
    "Expected_xA"
]

donor_merge[selected_numeric_cols].dtypes

,0
Touches_Touches,float64
Touches_Live,float64
Touches_Def 3rd,float64
Touches_Mid 3rd,float64
Touches_Att 3rd,float64
Touches_Att Pen,float64
Receiving_Rec,float64
Receiving_PrgR,float64
Progression_PrgR,float64
Total_Att,float64


In [ ]:
display(unmatched_full)

,player_norm,born_norm,season_norm,club_norm_merge
1162,bobby reid,1993,2022-2023,fulham
1243,mohammed daramy,2002,2023-2024,reims
1350,bobby reid,1993,2023-2024,fulham


In [ ]:
raw_donor_check = df2.copy()

raw_donor_check["season_norm"] = (
    raw_donor_check["season"]
    .apply(donor_season_to_standard)
)

raw_donor_check["born_norm"] = (
    pd.to_numeric(raw_donor_check["born_"], errors="coerce")
    .astype("Int64")
)

raw_donor_check["player_norm"] = (
    raw_donor_check["player"]
    .apply(normalize_text)
)

raw_donor_check["club_norm_merge"] = (
    raw_donor_check["team"]
    .apply(normalize_text)
)

In [ ]:
for _, row in unmatched_full.iterrows():

    candidates = raw_donor_check[
        (raw_donor_check["player_norm"] == row["player_norm"]) &
        (raw_donor_check["born_norm"] == row["born_norm"]) &
        (raw_donor_check["season_norm"] == row["season_norm"])
    ]

    print("=" * 100)
    print(
        row["player_norm"],
        row["born_norm"],
        row["season_norm"],
        row["club_norm_merge"]
    )

    if len(candidates) == 0:
        print("NO matching identity found in original donor dataset.")

    else:
        display(
            candidates[
                [
                    "player",
                    "born_",
                    "season",
                    "team",
                    "league",
                    "pos_",
                    "Playing Time_Min"
                ]
            ]
        )

bobby reid 1993 2022-2023 fulham
NO matching identity found in original donor dataset.
mohammed daramy 2002 2023-2024 reims
NO matching identity found in original donor dataset.
bobby reid 1993 2023-2024 fulham
NO matching identity found in original donor dataset.


## 7. Investigating Numeric Columns Imported as Text

Most selected donor statistics were imported as numeric fields.

Four expected numeric variables remained as `object`:

- `Standard_Dist`;
- `Expected_npxG/Sh`;
- `Expected_xAG`;
- `Expected_xA`.

Rather than coercing these immediately, their raw values are inspected first.

This is important because automatic coercion with `errors="coerce"` can silently convert valid but differently formatted values into missing data.

In [ ]:
object_numeric_cols = [
    "Standard_Dist",
    "Expected_npxG/Sh",
    "Expected_xAG",
    "Expected_xA"
]

for col in object_numeric_cols:
    print("=" * 80)
    print(col)
    print("=" * 80)

    print("dtype:", donor_merge[col].dtype)
    print("missing:", donor_merge[col].isna().sum())

    print("\nSample unique values:")
    print(donor_merge[col].dropna().unique()[:30])

    print()

Standard_Dist
dtype: object
missing: 2

Sample unique values:
['16,6' '18,4' '13,6' '19,9' '14,5' '27,7' 19 '18,7' '21,3' '24,1' '24,8'
 '16,3' '21,8' '15,9' '21,7' '27,3' '20,5' 23 '22,2' '14,6' '19,5' '21,5'
 '23,8' '15,4' '12,3' '17,6' '19,7' '22,8' '25,9' '19,2']

Expected_npxG/Sh
dtype: object
missing: 2

Sample unique values:
['0,11' '0,09' '0,18' '0,17' '0,04' '0,07' '0,14' '0,03' '0,05' '0,1'
 '0,06' '0,16' '0,19' '0,15' '0,08' '0,13' '0,24' '0,12' '0,23' '0,2'
 '0,21' '0,02' '0,22' '0,27' '0,25' '0,26' '0,48' '0,32' '0,31' '0,3']

Expected_xAG
dtype: object
missing: 0

Sample unique values:
['5,4' '3,5' '4,4' '4,9' '1,4' '3,4' '2,6' '8,7' '1,6' 1 '1,5' '2,2' '0,9'
 '3,3' '1,9' '3,2' '2,3' '8,3' '0,6' '1,1' '2,7' '4,7' '2,1' '0,4' '6,8'
 '6,4' '2,8' '2,9' '1,3' '1,2']

Expected_xA
dtype: object
missing: 0

Sample unique values:
['5,3' '3,6' 4 '6,4' '1,1' 5 '2,3' '7,7' '1,5' '1,2' '1,7' '1,6' '4,4'
 '1,8' '2,2' '2,8' '3,1' '1,3' '6,1' '0,5' 1 '5,1' '2,6' '0,4' '7,6' '7,4'
 '3,2'

In [ ]:
for col in object_numeric_cols:

    parsed = pd.to_numeric(
        donor_merge[col],
        errors="coerce"
    )

    bad_mask = (
        donor_merge[col].notna() &
        parsed.isna()
    )

    bad_values = (
        donor_merge.loc[bad_mask, col]
        .value_counts(dropna=False)
    )

    print("=" * 80)
    print(col)
    print("Non-numeric values:")
    print(bad_values)
    print()

Standard_Dist
Non-numeric values:
Standard_Dist
19,8    67
18,1    59
16,8    58
17,1    58
17,2    58
        ..
9,3      1
29,8     1
30,8     1
8,9      1
29,5     1
Name: count, Length: 211, dtype: int64

Expected_npxG/Sh
Non-numeric values:
Expected_npxG/Sh
0,07    539
0,06    518
0,08    493
0,05    479
0,09    457
0,1     424
0,04    412
0,11    399
0,12    344
0,13    303
0,14    249
0,03    225
0,15    213
0,16    170
0,17    121
0,18     99
0,19     67
0,2      56
0,21     36
0,02     23
0,22     18
0,23     11
0,24      9
0,25      4
0,26      4
0,29      3
0,28      3
0,27      2
0,3       2
0,31      1
0,32      1
0,48      1
0,33      1
Name: count, dtype: int64

Expected_xAG
Non-numeric values:
Expected_xAG
1,3     161
1,6     158
1,1     157
1,7     157
1,8     152
       ... 
10,6      1
14,8      1
12,7      1
9,7       1
12,8      1
Name: count, Length: 121, dtype: int64

Expected_xA
Non-numeric values:
Expected_xA
1,6     193
1,4     189
1,3     187
1,1     186
1,7 

### Comma Decimal Formatting

Inspection shows that these fields are not contaminated by arbitrary text.

Instead, decimal values are primarily represented using a comma separator, for example:

- `16,6` rather than `16.6`;
- `0,07` rather than `0.07`;
- `5,4` rather than `5.4`.

This explains why pandas imported the columns as `object`.

The comma separator is therefore standardized before numeric conversion.

Two missing values already exist in both `Standard_Dist` and `Expected_npxG/Sh`; these are genuine source-level missing values rather than conversion failures.

In [ ]:
object_numeric_cols = [
    "Standard_Dist",
    "Expected_npxG/Sh",
    "Expected_xAG",
    "Expected_xA"
]

def convert_comma_decimal(series):
    return pd.to_numeric(
        series.astype("string").str.replace(",", ".", regex=False),
        errors="coerce"
    )

for col in object_numeric_cols:
    donor_merge[col] = convert_comma_decimal(donor_merge[col])

In [ ]:
donor_merge[object_numeric_cols].dtypes

,0
Standard_Dist,Float64
Expected_npxG/Sh,Float64
Expected_xAG,Float64
Expected_xA,Float64


In [ ]:
donor_merge[object_numeric_cols].isna().sum()

,0
Standard_Dist,2
Expected_npxG/Sh,2
Expected_xAG,0
Expected_xA,0


### Numeric Conversion Result

After replacing comma decimal separators, all four fields convert successfully to numeric form.

The conversion introduces **no additional missing values**:

- `Expected_xAG`: 0 missing;
- `Expected_xA`: 0 missing;
- `Standard_Dist`: 2 missing;
- `Expected_npxG/Sh`: 2 missing.

The two missing values in the latter variables are retained rather than imputed.

At this stage there is no justification for inventing replacement values for variables that have not even been selected as modelling features.

## 8. Diagnosing the Remaining Identity Exceptions

A first search using the exact normalized player name found no corresponding donor identity for the three failed observations.

The matching criteria are therefore relaxed carefully.

Instead of using fuzzy matching, the next search uses:

> **birth year + season + club**

while temporarily ignoring player name.

This is a controlled diagnostic step.

If a clear candidate appears with the same birth year, club and season, the failure is likely a naming alias rather than a genuinely missing player.

In [ ]:
unmatched_candidates = []

for _, row in unmatched_full.iterrows():

    candidates = raw_donor_check[
        (raw_donor_check["born_norm"] == row["born_norm"]) &
        (raw_donor_check["season_norm"] == row["season_norm"]) &
        (raw_donor_check["club_norm_merge"] == row["club_norm_merge"])
    ].copy()

    print("=" * 100)
    print(
        f"Target: {row['player_norm']} | "
        f"{row['born_norm']} | "
        f"{row['season_norm']} | "
        f"{row['club_norm_merge']}"
    )

    if candidates.empty:
        print("No candidate found using birth year + season + club.")

    else:
        display(
            candidates[
                [
                    "player",
                    "born_",
                    "season",
                    "team",
                    "pos_",
                    "Playing Time_Min"
                ]
            ]
        )

Target: bobby reid | 1993 | 2022-2023 | fulham


,player,born_,season,team,pos_,Playing Time_Min
2864,Bobby De Cordova-Reid,1993.0,2223,Fulham,"FW,DF",2475


Target: mohammed daramy | 2002 | 2023-2024 | reims


,player,born_,season,team,pos_,Playing Time_Min
12932,Amir Richardson,2002.0,2324,Reims,MF,1352
12939,Josh Wilson-Esbrand,2002.0,2324,Reims,"DF,MF",461
12945,Mohamed Daramy,2002.0,2324,Reims,"FW,MF",1648


Target: bobby reid | 1993 | 2023-2024 | fulham


,player,born_,season,team,pos_,Playing Time_Min
3472,Bobby De Cordova-Reid,1993.0,2324,Fulham,FW,1423


### Identity Exceptions Identified

The relaxed search resolves all three failures.

Two distinct naming differences are responsible:

- `Bobby Reid` ↔ `Bobby De Cordova-Reid`;
- `Mohammed Daramy` ↔ `Mohamed Daramy`.

Bobby Reid appears in two seasons, which explains why two of the three failed observations come from the same underlying identity difference.

These are handled through explicit aliases applied only to the normalized merge field.

The original player-name columns remain unchanged.

This is preferable to introducing fuzzy matching across the entire dataset for only two known naming exceptions.

In [ ]:
player_name_aliases = {
    "bobby de cordova reid": "bobby reid",
    "mohamed daramy": "mohammed daramy"
}

donor_merge["player_norm"] = (
    donor_merge["player_norm"]
    .replace(player_name_aliases)
)

In [ ]:
full_keys = [
    "player_norm",
    "born_norm",
    "season_norm",
    "club_norm_merge"
]

full_match_check = (
    wingers_merge[full_keys]
    .drop_duplicates()
    .merge(
        donor_merge[full_keys].drop_duplicates(),
        on=full_keys,
        how="left",
        indicator=True
    )
)

print(full_match_check["_merge"].value_counts())

print(
    "\nMatch rate:",
    round(
        (full_match_check["_merge"] == "both").mean() * 100,
        2
    ),
    "%"
)

_merge
both          1396
left_only        0
right_only       0
Name: count, dtype: int64

Match rate: 100.0 %


### Final Match Coverage

After applying the two explicit aliases, the full player × club × season matching key achieves:

> **100% coverage — 1,396 of 1,396 observations**

At this point, identity resolution is considered complete.

No fuzzy-matching pipeline is required.

## 9. Enriching the Canonical Winger Dataset

The actual enrichment can now be performed.

The existing winger dataset remains the **left-hand authoritative dataset**, while the donor contributes additional columns.

The merge is performed using:

> **normalized player + birth year + season + club**

and explicitly validated as a **one-to-one** relationship.

A left join is used so that the donor source cannot add new players or redefine the previously established population.

In [ ]:
enriched_wingers = wingers_merge.merge(
    donor_merge,
    on=full_keys,
    how="left",
    suffixes=("", "_donor"),
    validate="one_to_one"
)

print("Original winger shape:", wingers_merge.shape)
print("Enriched shape:", enriched_wingers.shape)

Original winger shape: (1396, 81)
Enriched shape: (1396, 146)


In [ ]:
assert len(enriched_wingers) == len(wingers_merge)

print("Rows preserved:", len(enriched_wingers))

Rows preserved: 1396


### Merge Result

The merge preserves all:

> **1,396 canonical winger observations**

The enriched intermediate table contains **146 columns**.

Key donor fields such as touches, progressive receiving, passing attempts, crossing and total carries are present for every canonical observation.

The source-level missing values previously observed in `Standard_Dist` and `Expected_npxG/Sh` do not propagate into the final winger population, because the affected donor observation is outside the canonical winger dataset.

Identity coverage is therefore complete and the newly required behavioural fields are available for the full analytical population.

In [ ]:
donor_feature_cols = [
    "Touches_Touches",
    "Touches_Live",
    "Touches_Att 3rd",
    "Touches_Att Pen",
    "Receiving_Rec",
    "Receiving_PrgR",
    "Short_Att",
    "Medium_Att",
    "Long_Att",
    "Pass Types_Crs",
    "Carries_Carries",
    "Standard_Dist",
    "Expected_xAG"
]

enriched_wingers[donor_feature_cols].isna().sum()

,0
Touches_Touches,0
Touches_Live,0
Touches_Att 3rd,0
Touches_Att Pen,0
Receiving_Rec,0
Receiving_PrgR,0
Short_Att,0
Medium_Att,0
Long_Att,0
Pass Types_Crs,0


## 10. Cross-Source Consistency Audit

The overlapping donor columns can now serve their intended purpose.

For each duplicated statistic, the canonical value is compared directly against its donor equivalent.

This provides a stronger validation than identity matching alone.

If the wrong club-season record had been joined, statistics such as goals, shots, progressive carries and key passes would be unlikely to agree consistently.

The audit therefore tests both:

- merge correctness;
- cross-source consistency.

In [ ]:
# Original variable -> donor equivalent
audit_pairs = {
    "Goals": "Performance_Gls",
    "Assists": "Performance_Ast",
    "Progressive Carries": "Progression_PrgC",
    "Progressive Passes": "Progression_PrgP",
    "Key passes": "KP_",
    "Passes into Final 3rd": "1/3_",
    "Passes into Penalty Area": "PPA_",
    "Carries into Final 3rd": "Carries_1/3",
    "Carries into Penalty Area": "Carries_CPA",
    "Total Shots": "Standard_Sh"
}

audit_results = []

for original_col, donor_col in audit_pairs.items():

    original = pd.to_numeric(
        enriched_wingers[original_col],
        errors="coerce"
    )

    donor = pd.to_numeric(
        enriched_wingers[donor_col],
        errors="coerce"
    )

    # Handles floating-point values safely
    matches = np.isclose(
        original,
        donor,
        equal_nan=True
    )

    mismatch_count = (~matches).sum()

    audit_results.append({
        "original_column": original_col,
        "donor_column": donor_col,
        "rows": len(enriched_wingers),
        "matches": matches.sum(),
        "mismatches": mismatch_count,
        "match_%": round(matches.mean() * 100, 2)
    })

audit_summary = pd.DataFrame(audit_results)

display(audit_summary)

,original_column,donor_column,rows,matches,mismatches,match_%
0,Goals,Performance_Gls,1396,1396,0,100.00
1,Assists,Performance_Ast,1396,1395,1,99.93
2,Progressive Carries,Progression_PrgC,1396,1388,8,99.43
3,Progressive Passes,Progression_PrgP,1396,1393,3,99.79
4,Key passes,KP_,1396,1396,0,100.00
5,Passes into Final 3rd,1/3_,1396,1394,2,99.86
6,Passes into Penalty Area,PPA_,1396,1394,2,99.86
7,Carries into Final 3rd,Carries_1/3,1396,1387,9,99.36
8,Carries into Penalty Area,Carries_CPA,1396,1392,4,99.71
9,Total Shots,Standard_Sh,1396,1396,0,100.00


In [ ]:
audit_mismatches = {}

for original_col, donor_col in audit_pairs.items():

    original = pd.to_numeric(
        enriched_wingers[original_col],
        errors="coerce"
    )

    donor = pd.to_numeric(
        enriched_wingers[donor_col],
        errors="coerce"
    )

    mismatch_mask = ~np.isclose(
        original,
        donor,
        equal_nan=True
    )

    if mismatch_mask.any():

        mismatch_df = enriched_wingers.loc[
            mismatch_mask,
            [
                "player",
                "squad",
                "season",
                "Minutes",
                original_col,
                donor_col
            ]
        ].copy()

        audit_mismatches[f"{original_col} vs {donor_col}"] = mismatch_df

        print("=" * 100)
        print(f"{original_col}  VS  {donor_col}")
        print(f"Mismatches: {len(mismatch_df)}")

        display(mismatch_df.head(30))

Assists  VS  Performance_Ast
Mismatches: 1


,player,squad,season,Minutes,Assists,Performance_Ast
47,Douglas Costa,Juventus,2017-2018,1795,6,12


Progressive Carries  VS  Progression_PrgC
Mismatches: 8


,player,squad,season,Minutes,Progressive Carries,Progression_PrgC
656,Nabil Fekir,Betis,2020-2021,2622,103,104.0
661,Jorge de Frutos,Levante,2020-2021,2186,72,73.0
708,Stephy Mavididi,Montpellier,2020-2021,2030,76,78.0
1184,Gabriel Strefezza,Lecce,2022-2023,2442,72,71.0
1219,Dilane Bakwa,Strasbourg,2023-2024,2177,94,95.0
1221,Lameck Banda,Lecce,2023-2024,1350,70,71.0
1333,Remi Oudin,Lecce,2023-2024,1819,19,20.0
1377,Florian Thauvin,Udinese,2023-2024,1710,71,72.0


Progressive Passes  VS  Progression_PrgP
Mismatches: 3


...

,player,squad,season,Minutes,Passes into Penalty Area,PPA_
620,Ludovic Blas,Nantes,2020-2021,2787,26,25.0
777,Moses Simon,Nantes,2020-2021,2324,23,22.0


Carries into Final 3rd  VS  Carries_1/3
Mismatches: 9


,player,squad,season,Minutes,Carries into Final 3rd,Carries_1/3
620,Ludovic Blas,Nantes,2020-2021,2787,59,57.0
656,Nabil Fekir,Betis,2020-2021,2622,88,89.0
659,Fidel,Elche,2020-2021,2396,51,52.0
666,Bryan Gil,Eibar,2020-2021,2265,69,68.0
669,Amine Gouiri,Nice,2020-2021,2747,74,72.0
708,Stephy Mavididi,Montpellier,2020-2021,2030,35,36.0
777,Moses Simon,Nantes,2020-2021,2324,47,49.0
1171,Bukayo Saka,Arsenal,2022-2023,3181,92,93.0
1291,Khvicha Kvaratskhelia,Napoli,2023-2024,2737,108,109.0


Carries into Penalty Area  VS  Carries_CPA
Mismatches: 4


,player,squad,season,Minutes,Carries into Penalty Area,Carries_CPA
701,Rony Lopes,Nice,2020-2021,1783,20,19.0
777,Moses Simon,Nantes,2020-2021,2324,50,49.0
1219,Dilane Bakwa,Strasbourg,2023-2024,2177,24,25.0
1221,Lameck Banda,Lecce,2023-2024,1350,34,35.0


### Audit Findings

Agreement between the two independently scraped FBref-derived datasets is extremely high.

Several variables match perfectly:

- goals;
- key passes;
- total shots.

The remaining progression and carrying statistics disagree in only a very small number of observations.

Most discrepancies are only **±1 or ±2 actions over an entire season**, for example small differences in:

- progressive carries;
- progressive passes;
- passes into the final third;
- passes into the penalty area;
- carries into the final third;
- carries into the penalty area.

These differences are sufficiently small to be consistent with historical data revisions, different scrape dates or minor upstream corrections.

For these small disagreements, the original audited dataset remains canonical.

One discrepancy is qualitatively different:

> **Douglas Costa — Juventus — 2017–18: 6 assists in the original dataset vs 12 in the donor.**

A difference of six assists cannot reasonably be treated as minor source drift and therefore requires separate investigation.

In [ ]:
douglas_mask = (
    (enriched_wingers["player"] == "Douglas Costa") &
    (enriched_wingers["squad"] == "Juventus") &
    (enriched_wingers["season"] == "2017-2018")
)

display(
    enriched_wingers.loc[
        douglas_mask,
        [
            "player",
            "squad",
            "season",
            "Minutes",
            "Goals",
            "Assists",
            "Goals & Assists",
            "Goals per 90",
            "Assists per 90",
            "Performance_Gls",
            "Performance_Ast"
        ]
    ]
)

,player,squad,season,Minutes,Goals,Assists,Goals & Assists,Goals per 90,Assists per 90,Performance_Gls,Performance_Ast
47,Douglas Costa,Juventus,2017-2018,1795,4,6,10,0.2,0.3,4,12


### Correcting a Material Source Error: Douglas Costa

The Douglas Costa record is investigated independently rather than automatically trusting either dataset.

The donor value of **12 assists** is confirmed as the correct season total.

The original canonical dataset therefore contains a substantive error for this observation.

Because assists also affect derived variables, the correction is propagated to:

- `Assists`;
- `Goals & Assists`;
- `Assists per 90`.

No other source disagreements are automatically overwritten.

This is a targeted correction supported by a material discrepancy, not a decision to replace the original dataset wholesale with donor values.

In [ ]:
enriched_wingers.loc[douglas_mask, "Assists"] = 12

enriched_wingers.loc[douglas_mask, "Goals & Assists"] = (
    enriched_wingers.loc[douglas_mask, "Goals"] +
    enriched_wingers.loc[douglas_mask, "Assists"]
)

enriched_wingers.loc[douglas_mask, "Assists per 90"] = (
    enriched_wingers.loc[douglas_mask, "Assists"] /
    enriched_wingers.loc[douglas_mask, "Minutes"] * 90
).round(2)

In [ ]:
display(
    enriched_wingers.loc[
        douglas_mask,
        [
            "player",
            "squad",
            "season",
            "Minutes",
            "Goals",
            "Assists",
            "Goals & Assists",
            "Assists per 90",
            "Performance_Ast"
        ]
    ]
)

,player,squad,season,Minutes,Goals,Assists,Goals & Assists,Assists per 90,Performance_Ast
47,Douglas Costa,Juventus,2017-2018,1795,4,12,16,0.6,12


In [ ]:
print("wingers_df rows:", len(wingers_df))
print("wingers_merge rows:", len(wingers_merge))
print("enriched_wingers rows:", len(enriched_wingers))

wingers_df rows: 1396
wingers_merge rows: 1396
enriched_wingers rows: 1396


In [ ]:
canonical_duplicates = enriched_wingers[
    enriched_wingers.duplicated(
        subset=["player_id", "squad", "season"],
        keep=False
    )
].sort_values(
    ["player_id", "season", "squad"]
)

print(
    "Duplicate player_id × club × season rows:",
    len(canonical_duplicates)
)

display(
    canonical_duplicates[
        [
            "player",
            "player_id",
            "squad",
            "season",
            "Minutes"
        ]
    ]
)

Duplicate player_id × club × season rows: 0


,player,player_id,squad,season,Minutes


In [ ]:
merge_key_duplicates = enriched_wingers[
    enriched_wingers.duplicated(
        subset=[
            "player_norm",
            "born_norm",
            "season_norm",
            "club_norm_merge"
        ],
        keep=False
    )
].sort_values(
    [
        "player_norm",
        "season_norm",
        "club_norm_merge"
    ]
)

print(
    "Duplicate normalized merge-key rows:",
    len(merge_key_duplicates)
)

display(
    merge_key_duplicates[
        [
            "player",
            "player_id",
            "born",
            "squad",
            "season",
            "Minutes",
            "position_source"
        ]
    ]
)

Duplicate normalized merge-key rows: 0


,player,player_id,born,squad,season,Minutes,position_source


### Reconfirming the Canonical Population

After the correction, the dataset is recounted and checked again at the established observation grain.

The canonical population contains:

> **1,396 player × club × season observations**

with:

> **0 duplicate `player_id × squad × season` observations**

and:

> **0 duplicate normalized merge-key observations**

This also resolves the earlier apparent difference between 1,395 and 1,396 during exploratory matching: the three-field player-season audit collapsed one valid multi-club season, whereas the canonical project grain preserves club spells separately.

In [ ]:
print("Final enriched dataset shape:", enriched_wingers.shape)

assert len(enriched_wingers) == 1396, \
    "Unexpected row count after enrichment."

assert enriched_wingers.duplicated(
    subset=["player_id", "squad", "season"]
).sum() == 0, \
    "Duplicate player × club × season observations found."

assert enriched_wingers["player"].isna().sum() == 0
assert enriched_wingers["squad"].isna().sum() == 0
assert enriched_wingers["season"].isna().sum() == 0
assert enriched_wingers["player_id"].isna().sum() == 0

print("Final structural validation passed.")

Final enriched dataset shape: (1396, 146)
Final structural validation passed.


In [ ]:
audit_summary

,original_column,donor_column,rows,matches,mismatches,match_%
0,Goals,Performance_Gls,1396,1396,0,100.00
1,Assists,Performance_Ast,1396,1395,1,99.93
2,Progressive Carries,Progression_PrgC,1396,1388,8,99.43
3,Progressive Passes,Progression_PrgP,1396,1393,3,99.79
4,Key passes,KP_,1396,1396,0,100.00
5,Passes into Final 3rd,1/3_,1396,1394,2,99.86
6,Passes into Penalty Area,PPA_,1396,1394,2,99.86
7,Carries into Final 3rd,Carries_1/3,1396,1387,9,99.36
8,Carries into Penalty Area,Carries_CPA,1396,1392,4,99.71
9,Total Shots,Standard_Sh,1396,1396,0,100.00


In [ ]:
print("Rows:", enriched_wingers.shape[0])
print("Columns:", enriched_wingers.shape[1])

print("\nDuplicate player × club × season observations:")

duplicate_check = enriched_wingers.duplicated(
    subset=["player", "squad", "season"],
    keep=False
)

print(duplicate_check.sum())

Rows: 1396
Columns: 146

Duplicate player × club × season observations:
0


In [ ]:
key_donor_fields = [
    "Touches_Touches",
    "Touches_Live",
    "Touches_Att 3rd",
    "Touches_Att Pen",
    "Receiving_Rec",
    "Receiving_PrgR"
]

print(
    enriched_wingers[key_donor_fields]
    .isna()
    .sum()
)

Touches_Touches    0
Touches_Live       0
Touches_Att 3rd    0
Touches_Att Pen    0
Receiving_Rec      0
Receiving_PrgR     0
dtype: int64


### Closing the Merge Audit

The structural validation passes successfully.

The displayed cross-source audit table is intentionally retained as evidence of the **pre-correction comparison**, including the Douglas Costa discrepancy that triggered further investigation.

The small remaining historical differences are documented rather than forcibly reconciled.

At this point:

- identity matching is complete;
- all canonical rows have donor coverage;
- observation grain is preserved;
- the material assist error has been corrected;
- donor variables have been validated against overlapping canonical statistics.

The enrichment itself is therefore complete.

## 11. Preparing the Final Analytical Dataset

The enriched intermediate table still contains temporary infrastructure that was useful during matching and validation but should not remain in the final analytical dataset.

Cleanup removes:

- temporary normalized merge fields;
- redundant donor copies of canonical statistics;
- duplicated donor identity/context fields;
- merge-generated suffix columns.

The principle is:

> **Retain genuinely new analytical information; remove temporary matching infrastructure and redundant copies.**

A new `final_wingers` object is created so that the fully audited intermediate dataset remains untouched.

In [ ]:
final_wingers = enriched_wingers.copy()

In [ ]:
merge_helper_cols = [
    "player_norm",
    "born_norm",
    "season_norm",
    "club_norm_merge"
]

final_wingers = final_wingers.drop(
    columns=[
        col for col in merge_helper_cols
        if col in final_wingers.columns
    ]
)

In [ ]:
redundant_donor_cols = [
    "Performance_Gls",
    "Performance_Ast",
    "Progression_PrgC",
    "Progression_PrgP",
    "KP_",
    "1/3_",
    "PPA_",
    "Carries_1/3",
    "Carries_CPA",
    "Standard_Sh"
]

final_wingers = final_wingers.drop(
    columns=[
        col for col in redundant_donor_cols
        if col in final_wingers.columns
    ]
)

In [ ]:
possible_donor_identity_cols = [
    "player_donor",
    "season_donor",
    "league",
    "team",
    "nation_",
    "pos_",
    "age_",
    "born_",
    "Playing Time_MP",
    "Playing Time_Starts",
    "Playing Time_Min",
    "Playing Time_90s"
]

[
    col for col in possible_donor_identity_cols
    if col in final_wingers.columns
]

['player_donor',
 'season_donor',
 'league',
 'team',
 'nation_',
 'pos_',
 'age_',
 'born_',
 'Playing Time_MP',
 'Playing Time_Starts',
 'Playing Time_Min',
 'Playing Time_90s']

In [ ]:
donor_identity_cols_to_drop = [
    "player_donor",
    "season_donor",
    "league",
    "team",
    "nation_",
    "pos_",
    "age_",
    "born_",
    "Playing Time_MP",
    "Playing Time_Starts",
    "Playing Time_Min",
    "Playing Time_90s"
]

final_wingers = final_wingers.drop(
    columns=[
        col for col in donor_identity_cols_to_drop
        if col in final_wingers.columns
    ]
)

In [ ]:
donor_suffix_cols = [
    col for col in final_wingers.columns
    if col.endswith("_donor")
]

print(donor_suffix_cols)

['league_norm_donor']


In [ ]:
final_wingers = final_wingers.drop(
    columns=donor_suffix_cols
)

## 12. Final Schema Validation

An additional issue emerged when preparing the dataset for Parquet export.

CSV files are relatively permissive because values can ultimately be serialized as text. Parquet requires a more explicit and internally consistent schema.

The initial Parquet export exposed mixed Python types inside the take-on percentage columns.

For example, the same field contained:

- decimal values stored as strings;
- whole-number percentages stored as integers.

This is a consequence of the Excel-imported donor data rather than a football-data inconsistency.

The remaining `object` columns are therefore inspected explicitly before export.

In [ ]:
object_cols = final_wingers.select_dtypes(include="object").columns.tolist()

print(f"Object columns: {len(object_cols)}")

for col in object_cols:
    types = final_wingers[col].dropna().map(type).value_counts()

    if len(types) > 1:
        print(f"\n{col}")
        print(types)

Object columns: 19

Take-Ons_Succ%
Take-Ons_Succ%
<class 'str'>    1210
<class 'int'>     186
Name: count, dtype: int64

Take-Ons_Tkld%
Take-Ons_Tkld%
<class 'str'>    1210
<class 'int'>     186
Name: count, dtype: int64


In [ ]:
numeric_object_cols = [
    "Take-Ons_Succ%",
    "Take-Ons_Tkld%"
]

In [ ]:
for col in numeric_object_cols:

    if col in final_wingers.columns:

        final_wingers[col] = pd.to_numeric(
            final_wingers[col]
            .astype("string")
            .str.replace(",", ".", regex=False)
            .str.replace("%", "", regex=False)
            .str.strip(),
            errors="coerce"
        )

In [ ]:
final_wingers[numeric_object_cols].dtypes

,0
Take-Ons_Succ%,Float64
Take-Ons_Tkld%,Float64


In [ ]:
final_wingers[numeric_object_cols].isna().sum()

,0
Take-Ons_Succ%,0
Take-Ons_Tkld%,0


In [ ]:
remaining_object_cols = (
    final_wingers
    .select_dtypes(include="object")
    .columns
)

for col in remaining_object_cols:
    final_wingers[col] = final_wingers[col].astype("string")

In [ ]:
print(final_wingers.dtypes.value_counts())

float64           61
int64             34
string[python]    17
Float64            6
bool               1
Name: count, dtype: int64


In [ ]:
remaining_objects = final_wingers.select_dtypes(include="object").columns.tolist()

print("Remaining object columns:")
print(remaining_objects)

Remaining object columns:
[]


### Schema Cleaning Result

Two mixed-type numeric fields were identified:

- `Take-Ons_Succ%`;
- `Take-Ons_Tkld%`.

They were converted explicitly to numeric values after standardizing comma decimal separators.

No missing values were introduced.

All remaining genuine text fields were then converted from generic pandas `object` dtype to explicit string dtype.

The final table now contains no ambiguous `object` columns, giving the Parquet file a consistent schema for subsequent analysis.

## 13. Final Quality Checks

Before exporting the dataset, a final analytical QA pass is performed.

The checks verify:

- final dataset dimensions;
- duplicate player × club × season observations;
- season distribution;
- missingness in the most important newly added behavioural variables;
- final column names.

The objective is to ensure that the cleanup process did not accidentally remove observations, duplicate rows or damage the enriched variables.

In [ ]:
print("Final dataset shape:")
print(final_wingers.shape)

print("\nDuplicate player × club × season rows:")
print(
    final_wingers.duplicated(
        subset=["player", "squad", "season"]
    ).sum()
)

print("\nSeason distribution:")
print(
    final_wingers["season"]
    .value_counts()
    .sort_index()
)

print("\nTotal missing values in known enriched fields:")

check_cols = [
    "Touches_Touches",
    "Touches_Live",
    "Touches_Att 3rd",
    "Touches_Att Pen",
    "Receiving_Rec",
    "Receiving_PrgR",
    "Short_Att",
    "Medium_Att",
    "Long_Att",
    "Pass Types_Crs",
    "Carries_Carries",
    "Carries_PrgDist",
    "Carries_TotDist",
    "Standard_Dist",
    "Expected_npxG/Sh",
    "Expected_xAG",
    "Expected_xA"
]

display(
    final_wingers[check_cols]
    .isna()
    .sum()
    .to_frame("missing_values")
)

Final dataset shape:
(1396, 119)

Duplicate player × club × season rows:
0

Season distribution:
season
2017-2018    226
2018-2019    202
2019-2020    175
2020-2021    205
2021-2022    185
2022-2023    213
2023-2024    190
Name: count, dtype: int64

Total missing values in known enriched fields:


,missing_values
Touches_Touches,0
Touches_Live,0
Touches_Att 3rd,0
Touches_Att Pen,0
Receiving_Rec,0
Receiving_PrgR,0
Short_Att,0
Medium_Att,0
Long_Att,0
Pass Types_Crs,0


In [ ]:
print(f"Final rows: {final_wingers.shape[0]}")
print(f"Final columns: {final_wingers.shape[1]}")

for i, col in enumerate(final_wingers.columns, start=1):
    print(f"{i:>3}. {col}")

Final rows: 1396
Final columns: 119
  1. player
  2. nation
  3. pos
  4. squad
  5. comp
  6. age
  7. born
  8. Matches Played
  9. Minutes
 10. Goals
 11. Assists
 12. Goals & Assists
 13. Non-Penalty Goals
 14. Penalty Kicks Made
 15. Expected Goals
 16. Non-Penalty Expected Goals
 17. Progressive Carries
 18. Progressive Passes
 19. Goals per 90
 20. Assists per 90
 21. Tackles Attempted
 22. Tackles Won
 23. Dribblers Tackled %
 24. Shots blocked
 25. Passes blocked
 26. Interceptions
 27. Clearances
 28. Errors made
 29. Goals Against
 30. Goals Against per 90
 31. Saves
 32. Save %
 33. Clean Sheets
 34. Clean Sheet %
 35. Penalty Save %
 36. Passes Completed
 37. Passes Attempted
 38. Pass Completion %
 39. Progressive Passing Distance
 40. Short Pass Completion %
 41. Medium Pass Completion %
 42. Long Pass Completion %
 43. Key passes
 44. Passes into Final 3rd
 45. Passes into Penalty Area
 46. Touches in Defensive Penalty Area
 47. Take-ons Attempted
 48. Successful Take-o

In [ ]:
display(final_wingers.head())

,player,nation,pos,squad,comp,age,born,Matches Played,Minutes,Goals,...,SCA Types_Def,SCA_SCA,Aerial Duels_Won,Aerial Duels_Lost,Tackles_Att 3rd,Take-Ons_Att,Take-Ons_Succ,Take-Ons_Succ%,Take-Ons_Tkld,Take-Ons_Tkld%
0,Marc Albrighton,England,MF,Leicester City,Premier League,27,1989,34,2538,2,...,0.0,79.0,11.0,15.0,7.0,95.0,52.0,"54,7",43.0,"45,3"
1,Romain Amalfitano,France,MF,Dijon,Ligue 1,27,1989,33,2703,0,...,0.0,53.0,13.0,29.0,3.0,71.0,47.0,"66,2",24.0,"33,8"
2,Amath,Senegal,MF,Getafe,La Liga,21,1996,37,2859,3,...,1.0,61.0,19.0,47.0,18.0,98.0,43.0,"43,9",55.0,"56,1"
3,Nordin Amrabat,Morocco,FW,Leganés,La Liga,30,1987,30,1623,2,...,0.0,53.0,12.0,23.0,4.0,71.0,40.0,"56,3",31.0,"43,7"
4,Felipe Anderson,Brazil,FW,Lazio,Serie A,24,1993,21,1162,4,...,1.0,84.0,3.0,12.0,16.0,114.0,77.0,"67,5",37.0,"32,5"


In [ ]:
assert len(final_wingers) == 1396, \
    "Unexpected row count after enrichment."

assert final_wingers.duplicated(
    subset=["player", "squad", "season"]
).sum() == 0, \
    "Duplicate player × club × season observations found."

assert final_wingers["player"].isna().sum() == 0
assert final_wingers["season"].isna().sum() == 0
assert final_wingers["squad"].isna().sum() == 0

print("Final dataset validation passed.")

Final dataset validation passed.


## Final Dataset

The completed enriched dataset contains:

> **1,396 player × club × season observations**

and:

> **119 columns**

The established winger population remains unchanged.

The dataset now supplements the original audited FBref variables with additional information covering:

- touch volume and territorial involvement;
- attacking penalty-area occupation;
- receiving and progressive receiving;
- passing-length choices;
- crossing, through balls and switches;
- overall carrying involvement and carrying distance;
- shot selection;
- expected chance creation;
- shot-creating-action mechanisms;
- aerial involvement;
- selected high-pitch defensive activity.

Importantly, the presence of a variable in this dataset does **not** imply that it will enter the final playstyle model.

This notebook answers:

> **What reliable information do we have available for the winger population?**

It deliberately does **not** answer:

> **Which variables should define winger playstyle?**

That is the purpose of the next notebook.

In [ ]:
csv_path = "fbref_2017_2024_wingers_enriched.csv"

final_wingers.to_csv(
    csv_path,
    index=False
)

print(f"Saved: {csv_path}")

Saved: fbref_2017_2024_wingers_enriched.csv


In [ ]:
parquet_path = "fbref_2017_2024_wingers_enriched.parquet"

final_wingers.to_parquet(
    parquet_path,
    index=False
)

print(f"Saved: {parquet_path}")

Saved: fbref_2017_2024_wingers_enriched.parquet


In [ ]:
csv_check = pd.read_csv(
    "fbref_2017_2024_wingers_enriched.csv"
)

parquet_check = pd.read_parquet(
    "fbref_2017_2024_wingers_enriched.parquet"
)

print("In-memory:", final_wingers.shape)
print("CSV:", csv_check.shape)
print("Parquet:", parquet_check.shape)

In-memory: (1396, 119)
CSV: (1396, 119)
Parquet: (1396, 119)


In [ ]:
assert csv_check.shape == final_wingers.shape
assert parquet_check.shape == final_wingers.shape

print("Export validation passed.")

Export validation passed.


# Notebook Conclusion

The final enriched dataset is exported as:

- `fbref_2017_2024_wingers_enriched.csv`
- `fbref_2017_2024_wingers_enriched.parquet`

The Parquet version will be preferred as the input to subsequent Python analysis because it preserves the cleaned schema and numeric data types.

This closes the data-enrichment stage.

The next notebook is:

# `04_playstyle_feature_engineering.ipynb`

That notebook will begin the actual construction of a winger behavioural representation.

The next methodological questions include:

- Which available variables are genuine **style descriptors**?
- Which variables mainly capture **quality, success or output**?
- Which seasonal totals should become **per-90 rates**?
- Which behaviours should also be expressed relative to touches or opportunities?
- Which features are mathematically or conceptually redundant?
- Which metrics remain stable enough across the seven-season period?
- Which engineered ratios provide real football meaning rather than additional complexity?

The objective will be to produce a defensible **style feature matrix** that describes *how a winger plays*, while keeping player quality and attacking output conceptually separate.